In [1]:
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
import polars as pl


In [2]:
df = pl.read_csv("./predic/out/csv/large_outputs.csv")

predictions = df["caption"].to_list()
references = [
    ref.strip().strip('"').strip("'") 
    for ref in df["targetcaption"].to_list()
]

gts = {i: [ref] for i, ref in enumerate(references)}
res = {i: [pred] for i, pred in enumerate(predictions)}

In [3]:
for i, (ref, pred) in enumerate(zip(references, predictions)):
  print(f"gts[{i}]: {gts[i]}")
  print(f"res[{i}]: {res[i]}")
  print("-" * 40)  # Just a separator for readability

gts[0]: ['Two young guys with shaggy hair look at their hands while hanging out in the yard .']
res[0]: ['Ly dressed men are standing in front of a building with graffiti on it . one man is holding a bottle and the other two are wearing jeans and t-shirts . they look like they are about to']
----------------------------------------
gts[1]: ['Several men in hard hats are operating a giant pulley system .']
res[1]: ['Members of the military are working on a project . .']
----------------------------------------
gts[2]: ['A child in a pink dress is climbing up a set of stairs in an entry way .']
res[2]: ['Little girl in a pink shirt and blue jeans is standing on the steps of a house with her arms outstretched .']
----------------------------------------
gts[3]: ['Someone in a blue shirt and hat is standing on stair and leaning against a window .']
res[3]: ['Is a man in a blue shirt and jeans standing on the roof of a building with his hands in his pockets .']
-----------------------------

In [4]:

# Helper function to compute scores
def evaluate_coco(gts, res):
  scorers = [
      (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
      (Meteor(), "METEOR"),
      (Rouge(), "ROUGE_L"),
      (Cider(), "CIDEr"),
  ]

  eval_results = {}
  for scorer, method in scorers:
    score, scores = scorer.compute_score(gts, res)
    if isinstance(method, list):
      for m, s in zip(method, score):
        eval_results[m] = s
    else:
      eval_results[method] = score

  return eval_results

results = evaluate_coco(gts, res)
results

{'testlen': 6276, 'reflen': 5981, 'guess': [6276, 5962, 5648, 5334], 'correct': [1739, 373, 121, 44]}
ratio: 1.049322855709572


{'Bleu_1': 0.2770873167622248,
 'Bleu_2': 0.1316639116629912,
 'Bleu_3': 0.07187999912619449,
 'Bleu_4': 0.0418365596216047,
 'METEOR': 0.11774289511677875,
 'ROUGE_L': np.float64(0.238428961168091),
 'CIDEr': np.float64(0.2112347919762434)}